![logo](https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/docs/source/_figures/Helmholtz-AI.png?raw=true)

# Transformer Models

In this Notebook we will briefly introduce the concept of transformers in machine learning and show you how to train a transformer model.

---

## Getting Started

### Setup Colab environment

If you installed the packages and requirements on your  machine, you can skip this section and start from the import section.
Otherwise, you can follow and execute the tutorial on your browser. To start working on the notebook, click on the following button. This will open this page in the Colab environment, and you will be able to execute the code on your own.

<a href="https://colab.research.google.com/github/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/xai-for-transformer/1-Tutorial_Transformer_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now that you opened the notebook in Google Colab, follow the next step:

1. Run this cell to connect your Google Drive to Colab and install packages
2. Allow this notebook to access your Google Drive files. Click on 'Yes', and select your account.
3. "Google Drive for desktop wants to access your Google Account". Click on 'Allow'.
   
At this point, a folder has been created in your Drive, and you can navigate it through the lefthand panel in Colab. You might also receive an email that informs you about the access on your Google Drive.

In [1]:
# Mount drive folder to dbe abale to download repo
# from google.colab import drive
# drive.mount('/content/drive')

# Switch to correct folder'
# %cd /content/drive/MyDrive

In [2]:
# Don't run this cell if you already cloned the repo 
# %rm -r XAI-Tutorials
# !git clone --branch main https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials.git

In [3]:
# Install al required dependencies and package versions
# %cd XAI-Tutorials
# !pip install -r requirements_xai-for-transformer.txt
# %cd xai-for-transformer

### Imports

In [2]:
import math

import torch
import torch.nn as nn

import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

--------

## Build a Transformer Model

In the subsequent sections we will show you how to build a transformer architecture using PyTorch, we alternate a step-by-step implementaion with a small video explanation of the specific component.

***Note: we provide all references [here](https://xai-tutorials.readthedocs.io/en/latest/_ml_basics/transformer.html#references).***

### Motivation
Let's start with an introduction to the reason why transofmers became so popular and a visualization of the whole architecture

*TODO: Add video 1*

### Input Processing

Before an input sequence can be processed by a transformer model, it undergoes several preprocessing steps: 

- **Tokenization:** is a fundamental step in NLP. It involves splitting text into smaller units called tokens. Depending on the tokenization strategy, tokens can be words, subwords, characters or even an entire sentence.

  Example:  
  Text: "Natural Language Processing is fascinating."  
  Tokens: ["Natural", "Language", "Processing", "is", "fascinating"]  
  *TODO: add line of code* 

- **Input Embeddings:** After tokenization, each token is converted into a  high-dimensional vector representation called embedding. These embeddings wcapture semantic relationshios between tokens and serve as numerical input to the transformer.

    *TODO: add line of code + add video 2*


- **Positional Encodings:** Unlike recurrent or convolutional network, transformers do not inherently capture the order of tokens in a sequence. To provide positional information, positional encodings are added to the input embeddings. These encodings have the same dimensionality as the embeddings and are typically generated using sine and cosine functions of varying frequencies. This approach assigns a unique representation to each position, enabling the model to distinguish between tokens based on their order within the sequence.

    *TODO: add video 3*

In [ ]:
# Define the positional encoding layer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    

### Multi Head Attention

The Multi-Head Attention mechanism enables the Transformer to model relationships between different positions in a sequence. Instead of relying on a single attention operation, it uses multiple attention heads, each of which can focus on different aspects of the input and learn distinct patterns or dependencies.

Within each attention head, Scaled Dot-Product Attention is computed. The outputs of all attention heads are then concatenated and passed through a linear transformation to produce the final output representation.

Before diving into the implementation, it is helpful to first develop an intuitive understanding of the attention mechanism. The following video provides a clear explanation of how attention works and why it is a key component of Transformer models:

Attention Mechanism Explained: *TODO: Insert video link*

For a deeper understanding of the attention mechanisms used in the Transformer decoder, including Masked Self-Attention and Cross-Attention, you can also watch: (TODO: I would rather mention this after with the decoder part)

Attention in the Decoder (Masked and Cross-Attention): *TODO: Insert video link*

After reviewing these concepts, we will examine the code implementation of Multi-Head Attention in more detail.

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

### Putting It All Together

The original Transformer architecture consists of two main components: an **encoder** and a **decoder**. While many modern Transformer-based models use only one of these components, depending on the task they are designed to perform, it is useful to first understand the complete architecture before exploring specific applications.

*TODO: add video*

#### Encoder

The encoder processes the input sequence and transforms it into a contextualized representation that captures the relationships between tokens.

It is composed of a stack of identical layers, each containing:

* Multi-Head Self-Attention, which allows each token to attend to all other tokens in the sequence.
* Feed-Forward Network, which independently refines the representation of each token.
* Residual Connections and Layer Normalization, which improve training stability and help information flow through the network.

The output of the final encoder layer is a sequence of vectors that encodes the meaning and context of the input sequence. This representation can then be used by the decoder or by downstream tasks.


In [5]:
# The position-wise fully connected feed-forward network
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Define the encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

#### Decoder

The decoder generates the output sequence one token at a time based on the encoded input and previously generated tokens.

Each decoder layer contains:

* Masked Multi-Head Self-Attention, which prevents tokens from attending to future positions.
* Encoder-Decoder Attention (Cross-Attention), which allows the decoder to focus on relevant parts of the encoder output.
* Feed-Forward Network, which further processes the token representations.
* Residual Connections and Layer Normalization, which improve training stability.

The final decoder output is passed through a linear layer and a softmax function to produce probabilities over the vocabulary and generate the next token in the sequence.

In this course, we focus on Transformer models for understanding and classifying text. Therefore, we will mainly work with encoder-based architectures (e.g., BERT) and will not use the decoder component, which is primarily designed for sequence generation tasks such as machine translation and text generation.

If you are interested in the implementation details and the fine-tuning process for a machine translation task, please refer to the accompanying notebook:

TODO: add link to translation notebook.

### Assemble the final Transformer Classifier

For the final Transformer classifier, we combine the positional encodings, multi-head self-attention mechanisms, and stacked encoder layers introduced above into a complete encoder-only architecture suitable for text classification.

- **Mask generation:** An attention mask is created for the input sequence to prevent the attention mechanism from focusing on padding tokens. This ensures that only meaningful tokens contribute to the learned representations.

- **Embedding and positional encoding:** The input tokens are mapped to dense vector representations through an embedding layer (`embedding`). A `PositionalEncoding` layer is then applied to inject positional information, enabling the model to capture the order of tokens within the sequence.

- **Encoder layers:** The model consists of multiple encoder layers, defined by `num_layers`. Each encoder layer contains multi-head self-attention, feed-forward networks, residual connections, and layer normalization. The input embeddings are progressively transformed into contextual representations that capture relationships between all tokens in the sequence.

- **Sequence representation and classification head:** After the encoder stack, the contextual representation of the input sequence is aggregated into a sequence-level representation. This representation is then passed through a fully connected classification head to predict the output class (e.g., positive or negative sentiment).

> **Note:** Our simplified implementation uses the representation of the first token as a sequence-level representation. In BERT and DistilBERT, a special classification token (`[CLS]`) is prepended to every input sequence, and its final contextual representation is used by the classification head.

In [9]:
# Define the transformer classifier model
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers,
                 d_ff, max_seq_length, num_classes, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, input_ids):
        return (input_ids != 0).unsqueeze(1).unsqueeze(2)

    def forward(self, input_ids):
        mask = self.generate_mask(input_ids)

        x = self.embedding(input_ids)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        for encoder_layer in self.encoder_layers:
            x = encoder_layer(x, mask)

        cls_representation = x[:, 0, :]
        logits = self.classifier(cls_representation)

        return logits

### Parametrize the Transformer Classifier

To instantiate the Transformer classifier, we define the main architectural parameters that control the size and complexity of the model:

- `vocab_size`: The size of the vocabulary, i.e., the total number of unique tokens (words or subwords) that the model can process.
- `d_model`: The dimensionality of the token embeddings and hidden representations throughout the model.
- `num_heads`: The number of attention heads used in each multi-head self-attention layer.
- `num_layers`: The number of stacked encoder layers.
- `d_ff`: The dimensionality of the hidden layer in the feed-forward network contained in each encoder layer.
- `max_seq_length`: The maximum input sequence length supported by the positional encoding.
- `num_classes`: The number of output classes for the classification task.
- `dropout`: The dropout probability used as a regularization technique to reduce overfitting.

The following parameters define a small Transformer classifier for demonstration purposes.

In [ ]:

vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
num_classes = 2
dropout = 0.1


In [ ]:
transformer_classifier = TransformerClassifier(
    vocab_size,
    d_model,
    num_heads,
    num_layers,
    d_ff,
    max_seq_length,
    num_classes,
    dropout
).to(device)

## BERT

The previous sections introduced the original Transformer architecture and demonstrated how its main components can be assembled into an encoder-only Transformer classifier.

In practice, however, modern Natural Language Processing (NLP) applications rarely train Transformer models from scratch. Instead, they rely on **pretrained language models**, which are first trained on large text corpora and later adapted to specific downstream tasks through fine-tuning.

One of the most influential encoder-only Transformer models is **BERT (Bidirectional Encoder Representations from Transformers)**. Unlike the original encoder-decoder Transformer, BERT consists only of the encoder stack and is designed to learn contextual representations of text by attending to both the left and right context of every token simultaneously.

*TODO: add video*

In this course, we will use **DistilBERT**, a lightweight version of BERT that retains most of its performance while requiring fewer parameters and less computational resources. This makes it particularly suitable for educational purposes and explainability analyses.

*TODO: consider adding a table to show dofference between BERT and DistillBERT in readthedocs?*

### Loading a pretrained DistilBERT model

In [ ]:
checkpoint = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=6
    ).to(device)

## Extra Material: Finetune a Transformer Model (TODO: update for classification task)

In the subsequent sections we will show you how to fine-tune a transformer model using a pre-trained model from [Huggingface](https://huggingface.co/docs/transformers/index).

Our goal is to fine-tune a pre-trained seq2seq model from Huggingface for a translation task, i.e. translating english to german text.  
In this example we use the smaller version of the pre-trained Text-To-Text Transfer Transformer (T5) "T5-small" from google with 60 million parameters.  
You can find the model card [here](https://huggingface.co/google-t5/t5-small).  

In [12]:
checkpoint = "google-t5/t5-small"

The first step for fine-tuning a model is to retrieve a fine-tuning dataset and prepare it for the translation tasks.  
In our case we use the "opus_books" dataset, which contains parallel text in German (de) and English (en).  
In addition, we set the source language to English and the target language to German and define a task-specific prefix used by t5-small to understand the task, in this case, translating English text to German.

In [13]:
books = load_dataset("opus_books", "de-en")
books = books["train"].train_test_split(test_size=0.2)

source_lang = "en"
target_lang = "de"
prefix = "Translate English to German: "

Next, we initialize a tokenizer based on the specified model checkpoint, which will convert text into token IDs that the model can process  
and define a pre-processing function that prepares batches of translation examples for the translation task by:
- Prepending the task-specific prefix to the source language sentences.
- Extracting the corresponding target language sentences.
- Tokenizing both the inputs and targets into sequences of tokens suitable for input to the t5-small model.

In [14]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(examples):
    inputs = [prefix + example[source_lang] for example in examples["translation"]]
    targets = [example[target_lang] for example in examples["translation"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

tokenized_books = books.map(preprocess_function, batched=True)

Map:   0%|          | 0/41173 [00:00<?, ? examples/s]

Map:   0%|          | 0/10294 [00:00<?, ? examples/s]

For setting up the model we need to define the training arguments for our t5-small model using the `Seq2SeqTrainingArguments` class from the Hugging Face Transformers library.  
The training arguments include settings for the learning rate, batch size, number of epochs, evaluation strategy, model checkpointing, etc.

In [15]:
training_args = Seq2SeqTrainingArguments(
    output_dir="german-english",
    eval_strategy="epoch",
    learning_rate=2e-3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=False, 
    push_to_hub=False,
)

In addition to the tarining arguments, we need to define the evaluation metrics for our model.  
Here, we use the SacreBLEU metric, which is a standard evaluation metric for machine translation tasks,  
providing a BLEU (Bilingual Evaluation Understudy) score, which measures the quality of text generated by the model compared to reference translations.

In [16]:
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens) # average number of non-pad tokens in predictions (model output)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

Now we are finally ready to fine-tune the T5-small for our translation task.  
Therefore, we initializes the model using the pre-trained weights and configuration from the specified checkpoint.  
The data collator is responsible for dynamically padding the input sequences to the same length within a batch.  
This is important because batches often contain sequences of varying lengths.

After initilizing the Seq2SeqTrainer we can start the training/fine-tuning of the model.

*Note: for demonstration purpose we select only a small subsample of the train and test data.*

In [17]:
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_books["train"].select(range(200)), # we only select a small subsample for demonstration purposes
    eval_dataset=tokenized_books["test"].select(range(50)), # we only select a small subsample for demonstration purposes
    tokenizer=tokenizer, 
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/var/folders/8g/7wdchs993770j0nbzbt5p0_r0000gn/T/ipykernel_9213/688654782.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,No log,2.228585,3.361700,17.520000
2,No log,2.292926,3.867000,17.840000
3,No log,2.289472,4.270300,17.600000
4,No log,2.315202,4.109700,17.920000
5,No log,2.358456,4.277300,17.960000
6,No log,2.403290,3.727500,17.880000
7,No log,2.434193,3.710900,17.740000
8,No log,2.464301,3.830200,17.560000
9,No log,2.483917,3.844700,17.560000
10,No log,2.493277,3.851900,17.540000


TrainOutput(global_step=40, training_loss=1.6924976348876952, metrics={'train_runtime': 143.6457, 'train_samples_per_second': 13.923, 'train_steps_per_second': 0.278, 'total_flos': 65203027574784.0, 'train_loss': 1.6924976348876952, 'epoch': 10.0})